# Notebook 00 — Context

**Continual Learning in Context: RML Extension for CL-BENCH**

This notebook initializes the RML extension for CL-BENCH.

It establishes the basic lab-report workflow:

1. Load or synthesize stateful/stateless reward trajectories.
2. Compute gain.
3. Compare stateful vs. stateless performance.
4. Estimate stability and plasticity.
5. Flag possible stale-context / drift events.
6. Export reproducible figures, CSVs, and a downloadable zip archive.

This notebook is intentionally small. Later notebooks refine the analysis along the `{1, 7, 11, 13, 17, 19, 23, 29}` lane structure.

## 0. Colab / Local Setup

Run this cell first.

It supports both:

- local execution from `rml/notebooks/00_context.ipynb`
- Colab execution after cloning the repo

For Colab, run this before the setup cell if the repo is not already present:

```python
!git clone https://github.com/thinkthoughts/continual-learning-bench-rml.git
%cd continual-learning-bench-rml/rml
```

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()

# Common local case: .../rml/notebooks
if NOTEBOOK_DIR.name == "notebooks":
    RML_ROOT = NOTEBOOK_DIR.parent

# Common Colab / repo root case: .../continual-learning-bench-rml
elif (NOTEBOOK_DIR / "rml").exists() and (NOTEBOOK_DIR / "rml" / "src").exists():
    RML_ROOT = NOTEBOOK_DIR / "rml"

# Common case after: %cd continual-learning-bench-rml/rml
elif (NOTEBOOK_DIR / "src").exists():
    RML_ROOT = NOTEBOOK_DIR

# Fallback: search upward for an rml/src directory
else:
    RML_ROOT = None
    for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
        if (parent / "rml" / "src").exists():
            RML_ROOT = parent / "rml"
            break
        if (parent / "src").exists():
            RML_ROOT = parent
            break
    if RML_ROOT is None:
        raise RuntimeError(
            "Could not locate RML root. "
            "In Colab, run: !git clone https://github.com/thinkthoughts/continual-learning-bench-rml.git "
            "then %cd continual-learning-bench-rml/rml"
        )

if str(RML_ROOT) not in sys.path:
    sys.path.insert(0, str(RML_ROOT))

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook dir:", NOTEBOOK_DIR)
print("RML root:", RML_ROOT)
print("src exists:", (RML_ROOT / "src").exists())
print("results:", RESULTS_DIR)
print("figures:", FIGURES_DIR)

## 1. Imports

The imports below depend on the thin `rml/src` layer:

- `gain.py`
- `stability.py`
- `drift.py`
- `utils.py`

In [ ]:
import json
import zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.gain import compute_gain, cumulative_gain, average_gain
from src.stability import compute_stability, compute_plasticity
from src.drift import detect_drift
from src.utils import ensure_directory

print("Imports complete.")

## 2. Lab Report Context

CL-BENCH evaluates whether AI systems improve through sequential experience.

The benchmark separates:

- **reward**: how well the system performs on each task instance
- **gain**: how much the stateful system improves over the same system run statelessly

The core comparison is:

\[
g_t = r_t^{sf} - r_t^{sl}
\]

where:

- \(r_t^{sf}\) is stateful reward at instance \(t\)
- \(r_t^{sl}\) is stateless reward at instance \(t\)

RML adds an interpretability layer:

> What context was reused, when did it help, and when did it become stale?

## 3. Example Reward Trajectories

Notebook 00 uses a small synthetic trajectory so the extension runs immediately without requiring full CL-BENCH rollout logs.

Later notebooks can replace this dataframe with real CL-BENCH outputs.

In [ ]:
instances = np.arange(1, 13)

stateful_rewards = np.array([
    0.18, 0.22, 0.28, 0.35,
    0.43, 0.48, 0.46, 0.52,
    0.40, 0.45, 0.55, 0.60
])

stateless_rewards = np.array([
    0.18, 0.19, 0.20, 0.21,
    0.22, 0.20, 0.21, 0.22,
    0.23, 0.22, 0.24, 0.25
])

variant = [
    "A", "A", "A", "A",
    "B", "B", "B", "B",
    "C", "C", "C", "C"
]

df = pd.DataFrame({
    "instance": instances,
    "variant": variant,
    "stateful_reward": stateful_rewards,
    "stateless_reward": stateless_rewards,
})

df

## 4. Compute Gain

Gain measures the advantage of accumulated context over a fresh stateless run on the same instance.

In [ ]:
df["gain"] = compute_gain(
    df["stateful_reward"].tolist(),
    df["stateless_reward"].tolist(),
)

summary = {
    "cumulative_gain": cumulative_gain(df["gain"].tolist()),
    "average_gain": average_gain(df["gain"].tolist()),
    "mean_stateful_reward": float(df["stateful_reward"].mean()),
    "mean_stateless_reward": float(df["stateless_reward"].mean()),
}

summary

## 5. Stability and Plasticity

For this toy schedule, variants begin at instances:

- 1 → variant A
- 5 → variant B
- 9 → variant C

Using zero-based indices, those boundaries are:

```python
[0, 4, 8]
```

Interpretation:

- **stability**: gain at variant boundaries, where older structure must transfer
- **plasticity**: gain away from boundaries, where the system adapts within a variant

In [ ]:
variant_boundaries = [0, 4, 8]

stability = compute_stability(df["gain"].tolist(), variant_boundaries)
plasticity = compute_plasticity(df["gain"].tolist(), variant_boundaries)

summary["stability"] = stability
summary["plasticity"] = plasticity

summary

## 6. Drift / Stale Context Flags

A simple first-pass drift detector flags strongly negative gain.

In real CL-BENCH logs, negative gain can indicate:

- stale memory reuse
- overfitting to recent observations
- failure to detect concept drift
- harmful context compression

In [ ]:
# Default threshold is -0.10. Here we use a gentler threshold
# because the synthetic trajectory is small.
drift_indices = detect_drift(df["gain"].tolist(), threshold=-0.02)

df["possible_drift_or_stale_context"] = False
df.loc[drift_indices, "possible_drift_or_stale_context"] = True

df

## 7. Visualization — Stateful vs. Stateless

The first figure compares reward trajectories.

A widening gap means accumulated context is helping.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(df["instance"], df["stateful_reward"], marker="o", label="Stateful")
ax.plot(df["instance"], df["stateless_reward"], marker="o", linestyle="--", label="Stateless")

for boundary in [5, 9]:
    ax.axvline(boundary - 0.5, linestyle=":", linewidth=1)

ax.set_title("Notebook 00: Stateful vs. Stateless Reward")
ax.set_xlabel("Instance")
ax.set_ylabel("Reward")
ax.legend()
ax.grid(True, alpha=0.3)

reward_fig_path = FIGURES_DIR / "00_context_stateful_vs_stateless.png"
fig.tight_layout()
fig.savefig(reward_fig_path, dpi=160)

reward_fig_path

## 8. Visualization — Gain Trajectory

The second figure shows per-instance gain.

Positive gain means context helped. Negative gain means context may have hurt.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(df["instance"], df["gain"])
ax.axhline(0, linewidth=1)

for boundary in [5, 9]:
    ax.axvline(boundary - 0.5, linestyle=":", linewidth=1)

ax.set_title("Notebook 00: Gain Trajectory")
ax.set_xlabel("Instance")
ax.set_ylabel("Gain = Stateful - Stateless")
ax.grid(True, axis="y", alpha=0.3)

gain_fig_path = FIGURES_DIR / "00_context_gain.png"
fig.tight_layout()
fig.savefig(gain_fig_path, dpi=160)

gain_fig_path

## 9. Export Results

Notebook 00 saves:

- a CSV of the trajectory
- a JSON summary
- two PNG figures
- a zip archive for download or sharing

In [ ]:
csv_path = RESULTS_DIR / "00_context_gain.csv"
json_path = RESULTS_DIR / "00_context_summary.json"
zip_path = RESULTS_DIR / "00_context_artifacts.zip"

df.to_csv(csv_path, index=False)

summary_with_metadata = {
    **summary,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "00_context.ipynb",
    "extension": "continual-learning-bench-rml",
}

with open(json_path, "w") as f:
    json.dump(summary_with_metadata, f, indent=2)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in [csv_path, json_path, reward_fig_path, gain_fig_path]:
        z.write(path, arcname=path.name)

print("Saved:")
print("-", csv_path)
print("-", json_path)
print("-", reward_fig_path)
print("-", gain_fig_path)
print("-", zip_path)

## 10. Download Artifacts

In Colab, this cell exposes the zip as a browser download.

Locally, the archive is already saved under:

```text
rml/results/00_context_artifacts.zip
```

In [ ]:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as exc:
    print("Download helper is only active in Colab.")
    print("Artifact archive:", zip_path)

## 11. Notebook 00 Claim

CL-BENCH asks whether systems improve through sequential experience.

RML adds:

> Continual learning is not merely memory. It is context reuse under constraint.

Notebook 00 establishes the minimal workflow:

\[
\text{reward} \rightarrow \text{gain} \rightarrow \text{stability/plasticity} \rightarrow \text{drift flags}
\]

Later notebooks refine this with real CL-BENCH rollout logs and the eight-lane lab-report structure:

\[
0 \rightarrow \{1,7,11,13,17,19,23,29\}
\]